# Protein–DNA complex

Build a four-chain request containing two protein copies and a DNA duplex, then visualize the protein and nucleic acid chains.

Tested with **Boltz2 NIM 1.9.0 / client 0.6.0**.
From the repository root, install `python -m pip install -e ".[notebooks]"`
and start `jupyter lab examples/notebooks` in that environment.
Use a [ready NIM server](../../docs/deployment.md); `BOLTZ2_NIM_URL`
defaults to `http://localhost:8000`. Run cells in order.

[Setup and output files](../README.md#notebooks) ·
[NIM release notes](https://docs.nvidia.com/nim/bionemo/boltz2/1.9.0/release-notes.html)

Copyright (c) 2025-2026 NVIDIA Corporation.

In [ ]:
import os
from pathlib import Path
import boltz2_client
from boltz2_client import Boltz2Client, Polymer, PredictionRequest

client = Boltz2Client(
    base_url=os.getenv("BOLTZ2_NIM_URL", "http://localhost:8000"), timeout=1200,
)
metadata = await client.get_service_metadata()
print(f"Client {boltz2_client.__version__}; connected NIM {metadata.version} (tested: 1.9.0)")
output_dir = Path("notebook_outputs") / "protein_dna"
output_dir.mkdir(parents=True, exist_ok=True)

## Build a duplex with two protein chains

The protein construct and short DNA sequence demonstrate mixed-polymer input;
this notebook does not assign an experimentally validated interaction to them.
The protein includes a C-terminal His tag. DNA sequences are supplied in the
5′ → 3′ direction, with chain D calculated as the reverse complement of C.
All four chain IDs are unique.

In [ ]:
protein_sequence = 'MGREEPLNHVEAERQRREKLNQRFYALRAVVPNVSKMDKASLLGDAIAYINELKSKVVKTESEKLQIKNQLEEVKLELAGRLEHHHHHH'
dna_forward = "TGGGTCACGTGTTCC"
dna_reverse = dna_forward.translate(str.maketrans("ACGT", "TGCA"))[::-1]
print("DNA strands (both 5′ → 3′):", dna_forward, dna_reverse)

request = PredictionRequest(polymers=[
    Polymer(id="A", molecule_type="protein", sequence=protein_sequence),
    Polymer(id="B", molecule_type="protein", sequence=protein_sequence),
    Polymer(id="C", molecule_type="dna", sequence=dna_forward),
    Polymer(id="D", molecule_type="dna", sequence=dna_reverse),
])
result = await client.predict(request, output_dir=output_dir)

In [ ]:
print("Saved outputs:", output_dir.resolve())
print("Confidence:", result.confidence_scores)
print("Interface TM:", result.iptm_scores)

## View the complex

Protein chains A/B are shown as cyan/salmon cartoons; DNA chains C/D are shown
as blue/orange sticks. Confidence scores alone do not verify DNA recognition.

In [ ]:
import py3Dmol

view = py3Dmol.view(width=800, height=450)
view.addModel(result.structures[0].structure, "cif")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "cyan"}})
view.setStyle({"chain": "B"}, {"cartoon": {"color": "salmon"}})
view.setStyle({"chain": "C"}, {"stick": {"color": "blue"}})
view.setStyle({"chain": "D"}, {"stick": {"color": "orange"}})
view.zoomTo()
view.show()